# Capstone 2 — Data Wrangling (Weekly)

**Project:** Modeling USD/JPY movements with macroeconomic drivers

**Author:** Matt Snyder

This notebook extends the monthly pipeline (`01_data_wrangling.ipynb`) to a weekly frequency, per mentor feedback that monthly data leaves too few observations post-2006 for reliable modeling. Daily-native series (USD/JPY, Brent oil, USD index, Fed Funds) are re-pulled at their true native frequency and resampled to weekly. Lower-frequency series (CPI, GDP, JP call rate) are forward-filled, with a publication-lag shift applied first to avoid look-ahead bias.

## 1. Data Collection
Pull daily-native series via `pandas_datareader`, and load remaining monthly/quarterly series from `data_raw/`.

In [2]:
!pip install pandas-datareader

In [3]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
import datetime
import os

In [6]:
# Daily-native series — pull fresh via FRED API for reproducibility
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime.today()

usd_jpy_daily = web.DataReader('DEXJPUS', 'fred', start, end)
brent_oil_daily = web.DataReader('DCOILBRENTEU', 'fred', start, end)
dollar_indx_daily = web.DataReader('DTWEXBGS', 'fred', start, end)
fed_funds_daily = web.DataReader('DFF', 'fred', start, end)
us_cpi = web.DataReader('CPIAUCSL', 'fred', start, end)
us_gdp = web.DataReader('GDP', 'fred', start, end)
jp_call_rate = web.DataReader('IRSTCI01JPM156N', 'fred', start, end)

# Custom-built series — not available as a single FRED pull.
# Combines the discontinued FRED series (JPNCPIALLMINMEI, ends June 2021)
# with a manually rebased extension sourced from Japan's e-Stat, through present.
jp_cpi_extended = pd.read_csv("../data_processed/jp_cpi_extended.csv")

In [7]:
us_10y = web.DataReader('DGS10', 'fred', start, end)
vix = web.DataReader('VIXCLS', 'fred', start, end)
epu = web.DataReader('USEPUINDXD', 'fred', start, end)

In [8]:
jp_10y = web.DataReader('IRLTLT01JPM156N', 'fred', start, end)

In [11]:
epu.head(), epu.tail()

(            USEPUINDXD
 DATE                  
 2000-01-01       68.04
 2000-01-02      119.36
 2000-01-03       35.73
 2000-01-04      109.31
 2000-01-05      123.22,
             USEPUINDXD
 DATE                  
 2026-06-28      959.23
 2026-06-29      203.77
 2026-06-30      336.96
 2026-07-01      204.80
 2026-07-02      147.45)

In [16]:
usd_jpy_daily.head()
usd_jpy_daily.index.name

'DATE'

In [17]:
# Rename columns from FRED series IDs to readable names
usd_jpy_daily.rename(columns={'DEXJPUS': 'usd_jpy'}, inplace=True)
brent_oil_daily.rename(columns={'DCOILBRENTEU': 'brent_oil_price'}, inplace=True)
dollar_indx_daily.rename(columns={'DTWEXBGS': 'us_dollar_index'}, inplace=True)
fed_funds_daily.rename(columns={'DFF': 'us_fed_fund_rate'}, inplace=True)
us_cpi.rename(columns={'CPIAUCSL': 'us_cpi_index'}, inplace=True)
us_gdp.rename(columns={'GDP': 'us_gdp'}, inplace=True)
jp_call_rate.rename(columns={'IRSTCI01JPM156N': 'jp_call_rate'}, inplace=True)
us_10y.rename(columns={'DGS10': 'us_10y'}, inplace=True)
vix.rename(columns={'VIXCLS': 'vix'}, inplace=True)
epu.rename(columns={'USEPUINDXD': 'epu'}, inplace=True)
jp_10y.rename(columns={'IRLTLT01JPM156N': 'jp_10y'}, inplace=True)

In [20]:
usd_jpy_daily.head()

,usd_jpy
DATE,
2000-01-03,101.70
2000-01-04,103.09
2000-01-05,103.77
2000-01-06,105.19
2000-01-07,105.17


In [21]:
jp_10y.head()

,jp_10y
DATE,
2000-01-01,1.691
2000-02-01,1.796
2000-03-01,1.819
2000-04-01,1.740
2000-05-01,1.705


## 3. Resample to Weekly
Daily series resample directly. Lower-frequency series get a publication-lag shift applied before forward-filling, so no row contains data that wasn't actually public yet as of that week.

In [24]:
# TODO: resample daily series (.resample('W').last() or .mean() per variable)
# TODO: shift CPI/GDP by their publication lag, then .resample('W').ffill()

In [25]:
usd_jpy_w = usd_jpy_daily.resample('W').last()
brent_oil_w = brent_oil_daily.resample('W').mean()
dollar_indx_w = dollar_indx_daily.resample('W').last()
fed_funds_w = fed_funds_daily.resample('W').last()
us_10y_w = us_10y.resample('W').last()
vix_w = vix.resample('W').mean()
epu_w = epu.resample('W').mean()

In [26]:
# US CPI: released ~2-3 weeks after the month it covers
us_cpi_shifted = us_cpi.copy()
us_cpi_shifted.index = us_cpi_shifted.index + pd.Timedelta(weeks=3)
us_cpi_w = us_cpi_shifted.resample('W').ffill()

# US GDP: advance estimate released ~4 weeks after quarter-end
us_gdp_shifted = us_gdp.copy()
us_gdp_shifted.index = us_gdp_shifted.index + pd.Timedelta(weeks=4)
us_gdp_w = us_gdp_shifted.resample('W').ffill()

# JP call rate: monthly, minimal lag (policy rate, publicly known in near real-time)
jp_call_rate_w = jp_call_rate.resample('W').ffill()

# JP 10Y yield: monthly, minimal lag
jp_10y_w = jp_10y.resample('W').ffill()

# JP CPI (extended): released ~3-4 weeks after the month it covers, similar to US CPI
jp_cpi_extended['DATE'] = pd.to_datetime(jp_cpi_extended['DATE'])
jp_cpi_shifted = jp_cpi_extended.set_index('DATE')
jp_cpi_shifted.index = jp_cpi_shifted.index + pd.Timedelta(weeks=3)
jp_cpi_w = jp_cpi_shifted.resample('W').ffill()

In [29]:
jp_cpi_extended.head()

,DATE,jp_cpi_index
0,1955-01-01,17.09859
1,1955-02-01,17.09859
2,1955-03-01,16.98852
3,1955-04-01,17.13528
4,1955-05-01,16.97017


In [30]:
jp_cpi_shifted.head()

,jp_cpi_index
DATE,
1955-01-22,17.09859
1955-02-22,17.09859
1955-03-22,16.98852
1955-04-22,17.13528
1955-05-22,16.97017


## 4. Merge Datasets
Merge all series on a common weekly DATE index.

In [14]:
# TODO: merge all resampled series on DATE

In [59]:
# Organize by economic theme for readability/documentation - doesn't affect the merge itself
monetary_policy = [fed_funds_w, jp_call_rate_w, us_10y_w, jp_10y_w]
inflation = [us_cpi_w, jp_cpi_w]
growth = [us_gdp_w]
commodities = [brent_oil_w]
dollar_strength = [usd_jpy_w, dollar_indx_w]
risk_sentiment = [vix_w, epu_w]

merged_weekly = pd.concat(
    monetary_policy + inflation + growth + commodities + dollar_strength + risk_sentiment,
    axis=1
)

In [60]:
merged_weekly.head()

,us_fed_fund_rate,jp_call_rate,us_10y,jp_10y,us_cpi_index,jp_cpi_index,us_gdp,brent_oil_price,usd_jpy,us_dollar_index,vix,epu
DATE,,,,,,,,,,,,
1955-01-23,NaN,NaN,NaN,NaN,NaN,17.09859,NaN,NaN,NaN,NaN,NaN,NaN
1955-01-30,NaN,NaN,NaN,NaN,NaN,17.09859,NaN,NaN,NaN,NaN,NaN,NaN
1955-02-06,NaN,NaN,NaN,NaN,NaN,17.09859,NaN,NaN,NaN,NaN,NaN,NaN
1955-02-13,NaN,NaN,NaN,NaN,NaN,17.09859,NaN,NaN,NaN,NaN,NaN,NaN
1955-02-20,NaN,NaN,NaN,NaN,NaN,17.09859,NaN,NaN,NaN,NaN,NaN,NaN


In [63]:
merged_weekly['usd_jpy_target'] = merged_weekly['usd_jpy'].shift(-4)
merged_weekly.tail(10)

,us_fed_fund_rate,jp_call_rate,us_10y,jp_10y,us_cpi_index,jp_cpi_index,us_gdp,brent_oil_price,usd_jpy,us_dollar_index,vix,epu,usd_jpy_target
DATE,,,,,,,,,,,,,
2026-05-03,3.64,0.727,4.39,2.65,332.407,117.501604,NaN,119.6340,156.76,118.3926,17.708000,469.964286,159.23
2026-05-10,3.63,NaN,4.38,NaN,332.407,117.501604,NaN,105.8775,156.64,118.0392,17.466000,424.242857,160.26
2026-05-17,3.63,NaN,4.59,NaN,332.407,117.501604,NaN,110.5260,158.69,119.2825,17.986000,330.411429,160.24
2026-05-24,3.62,NaN,4.56,NaN,333.979,118.113060,NaN,110.6080,159.20,119.2868,17.356000,327.240000,161.37
2026-05-31,3.62,NaN,4.45,NaN,NaN,NaN,NaN,97.0525,159.23,118.8783,16.190000,295.250000,161.67
2026-06-07,3.62,NaN,4.55,NaN,NaN,NaN,NaN,98.9480,160.26,120.0831,16.958000,315.064286,NaN
2026-06-14,3.62,NaN,4.48,NaN,NaN,NaN,NaN,93.7640,160.24,119.5073,19.626000,261.444286,NaN
2026-06-21,3.63,NaN,4.46,NaN,NaN,NaN,NaN,81.0000,161.37,120.3958,16.846000,436.662857,NaN
2026-06-28,3.63,NaN,4.38,NaN,NaN,NaN,NaN,73.6340,161.67,120.8866,18.540000,431.260000,NaN


In [64]:
usd_jpy_daily.tail(10)

,usd_jpy
DATE,
2026-06-15,160.20
2026-06-16,160.39
2026-06-17,160.22
2026-06-18,161.37
2026-06-19,NaN
2026-06-22,161.51
2026-06-23,161.55
2026-06-24,161.76
2026-06-25,161.67


## 5. Data Quality Checks
Same checks as the monthly notebook: nulls, duplicates, date range, first/last valid date per column.

In [75]:
merged_weekly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3729 entries, 1955-01-23 to 2026-07-05
Freq: W-SUN
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   us_fed_fund_rate  1384 non-null   float64
 1   jp_call_rate      1375 non-null   float64
 2   us_10y            1383 non-null   float64
 3   jp_10y            1375 non-null   float64
 4   us_cpi_index      1371 non-null   float64
 5   jp_cpi_index      3723 non-null   float64
 6   us_gdp            1358 non-null   float64
 7   brent_oil_price   1383 non-null   float64
 8   usd_jpy           1382 non-null   float64
 9   us_dollar_index   1069 non-null   float64
 10  vix               1383 non-null   float64
 11  epu               1384 non-null   float64
 12  usd_jpy_target    1382 non-null   float64
dtypes: float64(13)
memory usage: 536.9 KB


In [65]:
merged_weekly.isnull().sum()

us_fed_fund_rate    2345
jp_call_rate        2354
us_10y              2346
jp_10y              2354
us_cpi_index        2358
jp_cpi_index           6
us_gdp              2371
brent_oil_price     2346
usd_jpy             2347
us_dollar_index     2660
vix                 2346
epu                 2345
usd_jpy_target      2347
dtype: int64

In [76]:
merged_weekly.index.duplicated().sum()

np.int64(0)

In [77]:
merged_weekly.index.min(), merged_weekly.index.max()

(Timestamp('1955-01-23 00:00:00'), Timestamp('2026-07-05 00:00:00'))

In [78]:
for col in merged_weekly.columns:
    first_valid = merged_weekly[col].first_valid_index()
    print(col, first_valid)

us_fed_fund_rate 2000-01-02 00:00:00
jp_call_rate 2000-01-02 00:00:00
us_10y 2000-01-09 00:00:00
jp_10y 2000-01-02 00:00:00
us_cpi_index 2000-01-23 00:00:00
jp_cpi_index 1955-01-23 00:00:00
us_gdp 2000-01-30 00:00:00
brent_oil_price 2000-01-09 00:00:00
usd_jpy 2000-01-09 00:00:00
us_dollar_index 2006-01-08 00:00:00
vix 2000-01-09 00:00:00
epu 2000-01-02 00:00:00
usd_jpy_target 1999-12-12 00:00:00


In [80]:
for col in merged_weekly.columns:
    last_valid = merged_weekly[col].last_valid_index()
    print(col, last_valid)

us_fed_fund_rate 2026-07-05 00:00:00
jp_call_rate 2026-05-03 00:00:00
us_10y 2026-07-05 00:00:00
jp_10y 2026-05-03 00:00:00
us_cpi_index 2026-05-24 00:00:00
jp_cpi_index 2026-05-24 00:00:00
us_gdp 2026-02-01 00:00:00
brent_oil_price 2026-07-05 00:00:00
usd_jpy 2026-06-28 00:00:00
us_dollar_index 2026-06-28 00:00:00
vix 2026-07-05 00:00:00
epu 2026-07-05 00:00:00
usd_jpy_target 2026-05-31 00:00:00


In [84]:
merged_weekly_model = merged_weekly[merged_weekly.index >= '2006-01-01'].copy()
merged_weekly_model.isnull().sum()

us_fed_fund_rate     0
jp_call_rate         9
us_10y               0
jp_10y               9
us_cpi_index        10
jp_cpi_index         6
us_gdp              22
brent_oil_price      0
usd_jpy              1
us_dollar_index      2
vix                  0
epu                  0
usd_jpy_target       5
dtype: int64

In [89]:
merged_weekly_model.shape

(1071, 13)

## 6. Save File

In [92]:
os.makedirs('../data_processed', exist_ok=True)

In [93]:
merged_weekly.index.name = 'DATE'
merged_weekly_model.index.name = 'DATE'

merged_weekly.to_csv('../data_processed/macro_weekly_merged.csv', index=True)
merged_weekly_model.to_csv('../data_processed/macro_weekly_merged_model.csv', index=True)